# Data Cleaning

In [1]:
%reset -f

Imports

In [4]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Loading the datasheets

In [5]:
DATA_RAW = Path("../data/raw")

studentInfo = pd.read_csv(DATA_RAW / "studentInfo.csv")
studentVle = pd.read_csv(DATA_RAW / "studentVle.csv")
assessments = pd.read_csv(DATA_RAW / "assessments.csv")
studentAssessment = pd.read_csv(DATA_RAW / "studentAssessment.csv")
courses = pd.read_csv(DATA_RAW / "courses.csv")
vle = pd.read_csv(DATA_RAW / "vle.csv")
studentReg = pd.read_csv(DATA_RAW / "studentRegistration.csv")

### Tables Overview

In [10]:
datasets = {
    "studentInfo": studentInfo,
    "studentVle": studentVle,
    "assessments": assessments,
    "studentAssessment": studentAssessment,
    "courses": courses,
    "vle": vle,
    "studentRegistration": studentReg
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name}")
    print(f"{'='*40}")
    print("Shape:", df.shape)
    print("\nMissing values:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())


studentInfo
Shape: (32593, 12)

Missing values:
 code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

Duplicate rows: 0

studentVle
Shape: (10655280, 6)

Missing values:
 code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

Duplicate rows: 787170

assessments
Shape: (206, 6)

Missing values:
 code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

Duplicate rows: 0

studentAssessment
Shape: (173912, 5)

Missing values:
 id_assessment       0
id_student          0
date_submitted      0

Handling Missing Values

In [19]:
studentAssessment["score"] = studentAssessment["score"].fillna(0)
studentInfo["imd_band"] = studentInfo["imd_band"].fillna("Unknown")

In [13]:
# Remove students who unregistered before the course started
studentReg = studentReg[~(studentReg['date_unregistration'] <= 0)]
#35 students have no date of registration and date of unregistration -to be handled later

Aggregating the studentVle sum\_click per day and merging the activity\_type to the student\_vle\_daily

In [16]:
# 1) Aggregate studentVle to daily level
studentVle_daily = (
    studentVle
    .groupby(["id_student","code_module","code_presentation","id_site","date"], as_index=False)
    .agg(sum_click=("sum_click","sum"))
)

# 2) Attach activity_type
vle_lookup = vle[['id_site', 'activity_type']].drop_duplicates()

studentVle_daily_enriched = studentVle_daily.merge(
    vle_lookup,
    on='id_site',
    how='left',
    validate='m:1'
)

# OPTIONAL but recommended for "course starts at day 0" modelling:
studentVle_daily_enriched = studentVle_daily_enriched[studentVle_daily_enriched['date'] >= 0].copy()

# 3) Week number
studentVle_daily_enriched['week'] = (studentVle_daily_enriched['date'] // 7) + 1

# 4) Map activity_type -> engagement_dim
CONTENT_TYPES = {'resource','url','page','book','folder','glossary','homepage','subpage','oucontent'}
ASSESSMENT_TYPES = {'quiz','questionnaire','externalquiz','assessment','assign'}
SOCIAL_TYPES = {'forumng','wiki','dataplus','collaborate','sharedsubpage'}

def map_engagement_dim(activity_type):
    if pd.isna(activity_type):
        return 'unknown'
    a = str(activity_type).strip().lower()
    if a in CONTENT_TYPES:
        return 'content'
    if a in ASSESSMENT_TYPES:
        return 'assessment'
    if a in SOCIAL_TYPES:
        return 'social'
    return 'other'

studentVle_daily_enriched['engagement_dim'] = studentVle_daily_enriched['activity_type'].apply(map_engagement_dim)
display(studentVle_daily_enriched)

,id_student,code_module,code_presentation,id_site,date,sum_click,activity_type,week,engagement_dim
0,6516,AAA,2014J,877011,204,6,forumng,30,social
1,6516,AAA,2014J,877011,209,8,forumng,30,social
2,6516,AAA,2014J,877011,218,9,forumng,32,social
3,6516,AAA,2014J,877012,0,38,forumng,1,social
4,6516,AAA,2014J,877012,5,12,forumng,1,social
...,...,...,...,...,...,...,...,...,...
8459315,2698588,BBB,2014J,985230,200,5,oucontent,29,content
8459316,2698588,BBB,2014J,985230,202,4,oucontent,29,content
8459317,2698588,BBB,2014J,985230,235,6,oucontent,34,content
8459318,2698588,BBB,2014J,992243,212,1,resource,31,content


In [20]:
# 5) Weekly clicks by engagement_dim (ALL weeks)
dim_clicks = (
    studentVle_daily_enriched
    .groupby(
        ['code_module', 'code_presentation', 'id_student', 'week', 'engagement_dim'],
        as_index=False
    )
    .agg(clicks=('sum_click', 'sum'))
)
# 6) Create total per student-course-week (sum across dimensions)
weekly_total = (
    dim_clicks
    .groupby(
        ['code_module','code_presentation','id_student','week'],
        as_index=False
    )
    .agg(clicks=('clicks','sum'))
)

weekly_total['engagement_dim'] = 'total'

# Combine dimension clicks + total clicks
dim_clicks = pd.concat([dim_clicks, weekly_total], ignore_index=True)

# 7) Create dynamic feature names like content_w1, content_w2, ..., total_wN
dim_clicks['feature_week'] = (
    dim_clicks['engagement_dim']
    + '_w'
    + dim_clicks['week'].astype(int).astype(str)
)

features_weekly_wide = dim_clicks.pivot_table(
    index=['code_module', 'code_presentation', 'id_student'],
    columns='feature_week',
    values='clicks',
    fill_value=0,
    aggfunc='sum'
).reset_index()


In [21]:
display(features_weekly_wide)

feature_week,code_module,code_presentation,id_student,assessment_w1,assessment_w10,assessment_w11,assessment_w12,assessment_w13,assessment_w14,assessment_w15,...,total_w36,total_w37,total_w38,total_w39,total_w4,total_w5,total_w6,total_w7,total_w8,total_w9
0,AAA,2013J,11391,0,0,0,0,0,0,0,...,7,1,0,0,0,26,60,22,20,0
1,AAA,2013J,28400,0,0,0,0,0,0,0,...,0,0,0,0,57,75,0,21,23,31
2,AAA,2013J,30268,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,AAA,2013J,31604,0,0,0,0,0,0,0,...,17,1,2,0,25,88,119,53,66,12
4,AAA,2013J,32885,0,0,0,0,0,0,0,...,7,0,0,0,58,18,16,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28495,GGG,2014J,2640965,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
28496,GGG,2014J,2645731,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,4,2,17
28497,GGG,2014J,2648187,1,0,0,0,0,16,0,...,0,0,0,0,5,5,0,0,0,1
28498,GGG,2014J,2679821,0,0,31,0,0,0,0,...,0,0,0,0,13,0,0,0,67,35


Demographics cleaning/encoding

In [24]:
studentInfo_clean = studentInfo.copy()

# Pass / Distinction = 0, Fail / Withdrawn = 1
studentInfo_clean['target_at_risk'] = (
    studentInfo_clean['final_result']
    .isin(['Fail', 'Withdrawn'])
    .astype(int)
)

studentInfo_clean['gender_bin'] = (
    studentInfo_clean['gender'].str.upper() == 'M'
).astype(int)

studentInfo_clean['disability_bin'] = (
    studentInfo_clean['disability'].str.upper() == 'Y'
).astype(int)

#one hot encoding done for age_band, highest education, region
age_ohe = pd.get_dummies(
    studentInfo_clean['age_band'],
    prefix='age',
    dummy_na=True
)

studentInfo_clean = pd.concat([studentInfo_clean, age_ohe], axis=1)

edu_ohe = pd.get_dummies(
    studentInfo_clean['highest_education'],
    prefix='edu',
    dummy_na=True
)

studentInfo_clean = pd.concat([studentInfo_clean, edu_ohe], axis=1)

def map_region_to_nation(region):
    if pd.isna(region):
        return 'Unknown'

    r = region.lower()

    if 'scotland' in r:
        return 'Scotland'
    if 'wales' in r:
        return 'Wales'
    if 'ireland' in r:
        return 'Northern_Ireland'
    if 'england' in r or 'region' in r:
        return 'England'

    return 'Other'

studentInfo_clean['nation'] = studentInfo_clean['region'].apply(map_region_to_nation)

nation_ohe = pd.get_dummies(
    studentInfo_clean['nation'],
    prefix='nation'
)

studentInfo_clean = pd.concat([studentInfo_clean, nation_ohe], axis=1)
# studentInfo_clean.drop(
#     columns=[
#         'gender', 'disability', 'age_band',
#         'highest_education', 'region', 'nation'
#     ],
#     inplace=True
# )

display(studentInfo_clean)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,edu_HE Qualification,edu_Lower Than A Level,edu_No Formal quals,edu_Post Graduate Qualification,edu_nan,nation,nation_England,nation_Northern_Ireland,nation_Scotland,nation_Wales
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,True,False,False,False,False,England,True,False,False,False
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,True,False,False,False,False,Scotland,False,False,True,False
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,False,False,False,False,False,England,True,False,False,False
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,False,False,False,False,False,England,True,False,False,False
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,False,True,False,False,False,England,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32588,GGG,2014J,2640965,F,Wales,Lower Than A Level,10-20,0-35,0,30,...,False,True,False,False,False,Wales,False,False,False,True
32589,GGG,2014J,2645731,F,East Anglian Region,Lower Than A Level,40-50%,35-55,0,30,...,False,True,False,False,False,England,True,False,False,False
32590,GGG,2014J,2648187,F,South Region,A Level or Equivalent,20-30%,0-35,0,30,...,False,False,False,False,False,England,True,False,False,False
32591,GGG,2014J,2679821,F,South East Region,Lower Than A Level,90-100%,35-55,0,30,...,False,True,False,False,False,England,True,False,False,False


In [27]:
KEYS = ['id_student', 'code_module', 'code_presentation']

# 1) Student info + registration (INNER)
info_reg = studentInfo_clean.merge(
    studentReg,
    on=KEYS,
    how='inner'
)

# 2) Add weekly engagement features (LEFT) to keep non-interacting students
master_table = info_reg.merge(
    features_weekly_wide,
    on=KEYS,
    how='left'
)

# 3) Fill missing engagement features with 0 (only feature columns, not the keys)
feature_cols = [c for c in features_weekly_wide.columns if c not in KEYS]
master_table[feature_cols] = master_table[feature_cols].fillna(0)

print(info_reg.shape, master_table.shape)


(29496, 32) (29496, 227)


In [33]:
ordered_cols = (
    [c for c in studentInfo_clean.columns if c not in KEYS] +
    [c for c in studentReg.columns if c not in KEYS] +
    [c for c in features_weekly_wide.columns if c not in KEYS]
)
master_table = master_table[KEYS + [c for c in ordered_cols if c in master_table.columns]]
display(master_table)

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,total_w36,total_w37,total_w38,total_w39,total_w4,total_w5,total_w6,total_w7,total_w8,total_w9
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,7.0,1.0,0.0,0.0,0.0,26.0,60.0,22.0,20.0,0.0
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,0.0,0.0,0.0,57.0,75.0,0.0,21.0,23.0,31.0
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,17.0,1.0,2.0,0.0,25.0,88.0,119.0,53.0,66.0,12.0
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,7.0,0.0,0.0,0.0,58.0,18.0,16.0,0.0,0.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29491,2640965,GGG,2014J,F,Wales,Lower Than A Level,10-20,0-35,0,30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29492,2645731,GGG,2014J,F,East Anglian Region,Lower Than A Level,40-50%,35-55,0,30,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,2.0,17.0
29493,2648187,GGG,2014J,F,South Region,A Level or Equivalent,20-30%,0-35,0,30,...,0.0,0.0,0.0,0.0,5.0,5.0,0.0,0.0,0.0,1.0
29494,2679821,GGG,2014J,F,South East Region,Lower Than A Level,90-100%,35-55,0,30,...,0.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,67.0,35.0


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=915c48ff-4287-4f39-af89-419738985e00' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>